**Fig. 5, step 2 (Fig. 5A, 5B): make the cell and nucleus masks consistent, every nucleus inside exactly one cell**

In [ ]:
# Working directory: data/fig5/ under the repository root, where the Fig. 5 deposit folder is downloaded
# (salivary_gland/ = SG.tif, cell_models/, nuclei_models/ and the engine's epicycle_output_cell/ and
# epicycle_output_nuclei/ trees; human_finetuning/; domain-specialized_finetuned_bootstrap/).
# Set the EPICYCLE_ANALYSES_DATA environment variable to keep the deposit somewhere else.
import os, sys
from pathlib import Path
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists() and (p / "data").is_dir()), None)
assert _root is not None, "start Jupyter from inside the EpiCycle-analyses repository"
os.chdir(Path(os.environ.get("EPICYCLE_ANALYSES_DATA", _root / "data")) / "fig5")
print("working directory:", os.getcwd())


# Cleanup → a consistent (cell, nucleus) consensus

This notebook makes the cell and nucleus segmentations **consistent** with each other and tidies them. It does not resolve merged cells (cells that hold two or more nuclei): those are kept as they are and reported. The cell labels are **trusted** and the nuclei are made to conform:

- each nucleus is assigned to the cell it mostly lies in. With `SPLIT_STRADDLING=True` (the setting used here) a nucleus that straddles two or more cells is **cut at the cell boundary**, so each cell keeps its own piece as a nucleus (the cell with the largest share keeps the original id; the other pieces get new ids). With `SPLIT_STRADDLING=False` the whole nucleus goes to the cell with the largest share and the other cell is left without a nucleus. A piece smaller than `MIN_PIECE_FRAC` of the nucleus's in-cell voxels is discarded, so a slight misalignment between the two masks does not create a sliver nucleus,
- holes in the cells are filled; signed-distance smoothing of the cells (the engine's smoother) is optional and off here (`SMOOTH = False`),
- a cell with **no nucleus** gets one recovered from the nuclear stain where possible (local rescale + threshold),
- cell fragments and nuclei outside every cell are dropped.

Cells with two or more nuclei are counted in the report and left unchanged. Output: a consistent `consensus_cells` + `consensus_nuclei` pair (`.npy` + BigTIFF, the same format as the engine's `global_labels`) in which every nucleus sits in exactly one cell.

The notebook works on any (cell, nucleus) mask pair, on a cluster or a single computer (`parallel=True/False`).

## 1. Paths  *(edit these)*

In [ ]:
from pathlib import Path
import numpy as np

CELL_RUN_DIR   = Path("salivary_gland/epicycle_output_cell")     # the CELL segmentation run
NUCLEI_RUN_DIR = Path("salivary_gland/epicycle_output_nuclei")   # the NUCLEI segmentation run
VOXEL_SIZE_UM  = (0.26, 0.26, 0.26)        # optional (z,y,x) um

# knobs
SMOOTH       = False         # signed-distance smoothing biases cell volume/shape stats -> keep OFF for a
                             # quantification mask (holes are STILL filled). Set True for viz only.
SIGMA        = 1.5
MIN_CELL_VOX = 100            # drop cell fragments smaller than this (0 = keep all)
RECOVER      = True          # recover missed nuclei in no-nucleus cells from the nuclear stain
SPLIT_STRADDLING = True      # CARVE: a nucleus straddling 2+ cells is CUT at the cell boundary so
                             # EACH cell keeps its own slice as a nucleus (majority keeps the
                             # original id; the others are relabeled). This is what lets every
                             # cell keep a nucleus. False = the alternative (the whole nucleus goes
                             # to the cell holding most of it; the other cell is left without a nucleus).
MIN_PIECE_FRAC = 0.2         # a slice must be >= this fraction of the nucleus's IN-CELL voxels to
                             # become its own nucleus; a smaller sliver is discarded (so small registration
                             # offsets between the masks do not create a sliver nucleus). Only used when SPLIT_STRADDLING.
SMOOTH_SPLIT_NUCLEI = True   # signed-distance smooth (SIGMA) ONLY the carved/split nucleus pieces (both halves)
                             # so the razor cut face rounds into a nucleus-like blob; clean + recovered
                             # nuclei are NOT touched. False = keep the flat cut. Carved ids are listed
                             # per cell in the triage `carved_nucleus_ids` -> exclude from nuclear
                             # SHAPE stats (the smoothed cut is a plausibility prior, not true shape).
NUCLEI_CHANNEL = 0           # which raw channel is the NUCLEAR stain (ZCYX array index — axis 1 of
                             # the canonicalized tile, so this is correct whether the ORIGINAL input
                             # was ZCYX or CZYX). 0 here = confirmed (on this data:
                             # nuclei = channel 0, membrane = channel 1). The membrane channel auto-detect
                             # (segmentation_params.channels) is unreliable on this run -> set
                             # explicitly. None = try auto (1 - membrane); re-confirm via a 2-channel
                             # render (see the diagnostic snippet) if you change datasets.

# --- compute resources ----
# Parallel workers MEMMAP the cell .npy once + pull only (id, bbox) tasks, so they actually
# saturate cores (no crop pickling through the parent). Biggest win when SMOOTH=True.
PARALLEL     = True          # use multiple cores for the per-cell pass
N_WORKERS    = None          # None = get_allocated_cpus()-1 (the CPUs allocated to this job,
                             # NOT the bare node count). Set an int to pin it.

CELL_LABELS_NPY   = CELL_RUN_DIR   / "stage4_full_stitch" / "global_labels_clean.npy"
NUCLEI_LABELS_NPY = NUCLEI_RUN_DIR / "stage4_full_stitch" / "global_labels_clean.npy"
CELL_TILES        = CELL_RUN_DIR   / "tiles"          # raw channels (for nucleus recovery)
for p in (CELL_LABELS_NPY, NUCLEI_LABELS_NPY):
    print(p, "OK" if p.exists() else "** MISSING **")


<run>/epicycle_output_cell/stage4_full_stitch/global_labels_clean.npy OK
<run>/epicycle_output_nuclei/stage4_full_stitch/global_labels_clean.npy OK


## 2. Carve + smooth → consistent pair  (no recovery yet)

In [ ]:
from packages.epicycle_scaledseg.reconcile import cleanup_volume

print("loading cell + nuclei label volumes (two multi-GB .npy reads — silent, ~tens of s)…", flush=True)
cells  = np.load(str(CELL_LABELS_NPY))      # full copy in RAM (needs about 60 GB of RAM)
nuclei = np.load(str(NUCLEI_LABELS_NPY))
assert cells.shape == nuclei.shape, f"cell {cells.shape} != nuclei {nuclei.shape} — not co-registered"
print(f"loaded. cells {cells.shape} {cells.dtype} (max id {int(cells.max()):,}) ; "
      f"nuclei max id {int(nuclei.max()):,}", flush=True)   # max() not unique() (no full sort)

# carve + majority-assign + (optional) smooth.  progress=True prints a [k/4] marker + tqdm bar for
# EVERY internal phase (assign / scan / clean / write-back).  Both *_path args memmap the .npy in
# the pool workers so the assign AND clean passes use cores.  recovery is the separate §3 pass.
# return_triage=True -> `triage`, one status row per surviving cell (drives the §6 feature table).
# split_straddling_nuclei -> CARVE mode: a straddling nucleus is split by the cell boundary (new
# status `split_nucleus`); min_piece_frac drops sub-threshold slivers.  smooth_split_nuclei rounds
# the razor cut on BOTH carved halves (only the split pieces; ids listed in triage['carved_nucleus_ids']).
# NOTE: with SMOOTH=False the per-cell work (fill-holes) is tiny + memmap-I/O-bound, so htop will show
# cores only lightly used — that's expected; the heavy CPU win is SMOOTH=True.
out_cells, out_nuclei, report, triage = cleanup_volume(
    cells, nuclei, nuclei_raw=None, smooth=SMOOTH, sigma=SIGMA,
    min_cell_vox=MIN_CELL_VOX, recover=True,
    split_straddling_nuclei=SPLIT_STRADDLING, min_piece_frac=MIN_PIECE_FRAC,
    smooth_split_nuclei=SMOOTH_SPLIT_NUCLEI,
    parallel=PARALLEL, n_workers=N_WORKERS,
    cells_path=str(CELL_LABELS_NPY), nuclei_path=str(NUCLEI_LABELS_NPY),
    progress=True, return_triage=True)

print("\n=== cleanup report ===")
for k, v in sorted(report.items()):
    print(f"  {k:22s} {v:,}")
print("verifying consistency (full-volume scan — silent)…", flush=True)
assert (out_nuclei[out_cells == 0] == 0).all(), "a nucleus sticks out of all cells!"
print("consistency OK: every nucleus is inside a cell.")
print(f"RESIDUAL (not resolved here): {report.get('merged_cell', 0):,} cells still hold 2+ nuclei (the merges).")

loading cell + nuclei label volumes (two multi-GB .npy reads — silent, ~tens of s)…
loaded. cells (266, 5631, 4709) int32 (max id 82,165) ; nuclei max id 80,169
[1/4] indexing nuclei (full-volume scan) + assigning each to its majority cell…


assign (23w):   0%|          | 0/80169 [00:00<?, ?it/s]

[2/4] indexing cells (full-volume scan) + bbox/fragment filter…


scan cells:   0%|          | 0/82165 [00:00<?, ?it/s]

[3/4] cleaning 82,063 cells (fill)…


cleanup (23w):   0%|          | 0/82063 [00:00<?, ?it/s]

[4/4] writing the consensus volumes…


write-back:   0%|          | 0/82063 [00:00<?, ?it/s]

<deps>/<engine>/core/smooth.py:95: FutureWarning: `skeletonize_3d` is deprecated since version 0.23 and will be removed in version 0.25. Use `skimage.morphology.skeletonize` instead.
  skeleton = skeletonize_3d(mask.astype(np.uint8)) > 0



=== cleanup report ===
  carved_nucleus_pieces  6,533
  clean                  72,319
  dropped_fragment       102
  dropped_nucleus_with_fragment 1
  merged_cell            1,867
  no_nucleus_genuine     3,436
  no_nucleus_strung_victim 1,406
  orphan_nucleus_dropped 4,163
  split_nucleus          3,035
verifying consistency (full-volume scan — silent)…
consistency OK: every nucleus is inside a cell.
RESIDUAL (not resolved here): 1,867 cells still hold 2+ nuclei (the merges).


## 3. Recover missed nuclei in the no-nucleus cells  *(optional)*

Per no-nucleus cell, read the **nuclear stain** at its bbox (via the read-only ROI reader — no full-volume load) and try to pull out a dim/deep nucleus by local contrast rescale + threshold. Recovered nuclei get fresh ids.

In [ ]:
if RECOVER:
    from scipy.ndimage import find_objects
    from packages.epicycle_scaledseg.reconcile import (read_global_roi, recover_nucleus, load_geometry,
                                    resolve_channel_layout)
    try:
        from tqdm.auto import tqdm
    except Exception:
        def tqdm(it, **k): return it
    geo = load_geometry(CELL_TILES); n_ch = geo['channels']
    lay = resolve_channel_layout(CELL_TILES)        # reads the pipeline's CHANNELS=[signal,nucleus]
    auto_nuc = lay.get('nucleus_channel')           # AUTHORITATIVE: segmentation_params.channels[1]-1
    # nuclei channel: explicit NUCLEI_CHANNEL override wins; else the config's nucleus channel
    if NUCLEI_CHANNEL is not None:
        nuc_ch = int(NUCLEI_CHANNEL)
    elif auto_nuc is not None:
        nuc_ch = int(auto_nuc)
    else:
        nuc_ch = None
    if n_ch < 2:
        print(f"cell run has {n_ch} channel(s) -> no nuclear channel; SKIPPING recovery.")
        RECOVER = False
    elif nuc_ch is None:
        print(f"no nuclei channel: config layout={lay}. The run pipeline's CHANNELS=[signal,nucleus] "
              f"isn't in this run's tile configs. Set NUCLEI_CHANNEL in §1 = CHANNELS[1]-1 "
              f"(on THIS data CHANNELS=[2,1] -> nuclei = channel 0). SKIPPING recovery for now.")
        RECOVER = False
    elif not (0 <= nuc_ch < n_ch):
        raise ValueError(f"NUCLEI_CHANNEL={nuc_ch} out of range for a {n_ch}-channel run")
    else:
        src = "NUCLEI_CHANNEL override" if NUCLEI_CHANNEL is not None else "segmentation config (CHANNELS[1])"
        assert out_cells.shape == (geo['depth'], geo['full'][0], geo['full'][1]), \
            "label volume != tile frame (not co-registered)"
        print(f"reading nuclear channel = {nuc_ch} (from {src}; signal/membrane = {lay.get('signal_channel')}). "
              f"Render both channels once to confirm channel {nuc_ch} is the bright nuclei blobs.")
        print("indexing cells for recovery (full-volume scan — silent)…", flush=True)
        has_nuc = np.zeros(int(out_cells.max()) + 1, dtype=bool)
        has_nuc[np.unique(out_cells[out_nuclei > 0])] = True
        Z, H, W = out_cells.shape; nxt = int(out_nuclei.max()) + 1; n_rec = 0
        tri_by_cell = {r["cell_id"]: r for r in triage}   # patch triage rows as nuclei are recovered
        # only the NO-NUCLEUS cells need a (slow) per-bbox stain read -> bar over just those
        targets = [(idx + 1, sl) for idx, sl in enumerate(find_objects(out_cells))
                   if sl is not None and not has_nuc[idx + 1]]
        print(f"{len(targets):,} no-nucleus cells to attempt recovery on (serial, per-cell disk read)")
        for cid, sl in tqdm(targets, desc="recovering nuclei"):
            z0, z1 = max(0, sl[0].start - 1), min(Z, sl[0].stop + 1)
            y0, y1 = max(0, sl[1].start - 2), min(H, sl[1].stop + 2)
            x0, x1 = max(0, sl[2].start - 2), min(W, sl[2].stop + 2)
            cmask = out_cells[z0:z1, y0:y1, x0:x1] == cid
            rec = recover_nucleus(read_global_roi(CELL_TILES, (z0, z1, y0, y1, x0, x1), nuc_ch), cmask)
            if rec is not None:
                out_nuclei[z0:z1, y0:y1, x0:x1][rec & cmask] = nxt
                r = tri_by_cell.get(cid)              # no_nucleus_* -> recovered_nucleus (keep triage true)
                if r is not None:
                    r["status"] = "recovered_nucleus"; r["n_nuclei"] = 1; r["nucleus_ids"] = [nxt]
                nxt += 1; n_rec += 1
        assert (out_nuclei[out_cells == 0] == 0).all(), "recovery wrote a nucleus outside a cell!"
        print(f"recovered {n_rec:,} missed nuclei. CAVEAT: recovery can inject false positives "
              f"(debris/bleed-in) — eyeball a sample before trusting. {len(targets) - n_rec:,} "
              f"cells still have no nucleus (kept + flagged).")
else:
    print("RECOVER=False — skipping nucleus recovery.")


reading nuclear channel = 0 (from NUCLEI_CHANNEL override; signal/membrane = None). Render both channels once to confirm channel 0 is the bright nuclei blobs.
indexing cells for recovery (full-volume scan — silent)…
4,842 no-nucleus cells to attempt recovery on (serial, per-cell disk read)


recovering nuclei:   0%|          | 0/4842 [00:00<?, ?it/s]

recovered 269 missed nuclei. CAVEAT: recovery can inject false positives (debris/bleed-in) — eyeball a sample before trusting. 4,573 cells still have no nucleus (kept + flagged).


In [ ]:
# --- Final tally: how many cells end up WITH a nucleus (after §2 cleanup + §3 recovery) ---
from collections import Counter as _Counter
if "triage" not in globals():
    print("no `triage` in memory — run §2 (and §3) first.")
else:
    _sc = _Counter(r["status"] for r in triage)
    _tot = len(triage)
    _HAS = ("clean", "split_nucleus", "recovered_nucleus", "merged_cell")   # >=1 nucleus
    _NONE = ("no_nucleus_strung_victim", "no_nucleus_genuine")               # 0 nuclei
    _has = sum(_sc.get(k, 0) for k in _HAS)
    _none = sum(_sc.get(k, 0) for k in _NONE)
    print("=== FINAL TALLY (after cleanup + recovery) ===")
    print(f"  {_tot:,} surviving cells")
    print(f"  {_has:,} have a nucleus ({100 * _has / max(_tot, 1):.1f}%)   |   "
          f"{_none:,} have none ({100 * _none / max(_tot, 1):.1f}%)")
    print("  breakdown by status:")
    for _k in _HAS + _NONE:
        print(f"     {_k:26s} {_sc.get(_k, 0):>10,}")
    print(f"  residual to curate (merged_cell = 2+ nuclei): {_sc.get('merged_cell', 0):,}")
    if "report" in globals():
        print(f"  (also dropped earlier in §2: {report.get('dropped_fragment', 0):,} fragment cells, "
              f"{report.get('orphan_nucleus_dropped', 0):,} orphan nuclei with no cell)")

=== FINAL TALLY (after cleanup + recovery) ===
  82,063 surviving cells
  77,490 have a nucleus (94.4%)   |   4,573 have none (5.6%)
  breakdown by status:
     clean                          72,319
     split_nucleus                   3,035
     recovered_nucleus                 269
     merged_cell                     1,867
     no_nucleus_strung_victim        1,309
     no_nucleus_genuine              3,264
  residual to curate (merged_cell = 2+ nuclei): 1,867
  (also dropped earlier in §2: 102 fragment cells, 4,163 orphan nuclei with no cell)


## 4. Write the consensus  *(set WRITE=True)*

In [ ]:
WRITE = True        # <-- set True to save the cleaned, consistent pair
if WRITE:
    import tifffile
    out_dir = CELL_RUN_DIR / "reconcile"; out_dir.mkdir(exist_ok=True)
    for name, vol in [("cell", out_cells), ("nuclei", out_nuclei)]:
        arr = np.ascontiguousarray(vol, dtype=np.int32)
        print(f"writing cleanup_{name}_labels (.npy + multi-GB BigTIFF — silent, can be minutes)…",
              flush=True)
        np.save(out_dir / f"cleanup_{name}_labels.npy", arr)
        tifffile.imwrite(str(out_dir / f"cleanup_{name}_labels.tif"), arr, bigtiff=True)
        print(f"  wrote cleanup_{name}_labels", flush=True)
    print(f"DONE. cleanup_cell_labels + cleanup_nuclei_labels (.npy + BigTIFF) in {out_dir}")
    print("every nucleus is inside exactly one cell; merged cells (2+ nuclei) remain as the residual.")
else:
    print("WRITE=False — nothing written. Review the report, then set True.")

writing cleanup_cell_labels (.npy + multi-GB BigTIFF — silent, can be minutes)…
  wrote cleanup_cell_labels
writing cleanup_nuclei_labels (.npy + multi-GB BigTIFF — silent, can be minutes)…
  wrote cleanup_nuclei_labels
DONE. cleanup_cell_labels + cleanup_nuclei_labels (.npy + BigTIFF) in <run>/epicycle_output_cell/reconcile
every nucleus is inside exactly one cell; merged cells (2+ nuclei) remain as the residual.


## 5. Inspect the consensus — interactive  *(optional)*

A quick inline look at the consensus. It uses matplotlib + ipywidgets (already in the env) and draws right here in the notebook output, which you already see in your browser over the Jupyter connection. Labels are **memory-mapped** from `reconcile/`; each raw slice is read **on demand** off the tiles (only the crop you're viewing).

Controls: **view** (XY / XZ / YZ — the axial views), **slice** (scroll through the perpendicular axis), **rows / cols** range sliders to **zoom into a region** (reads only that crop), per-layer **checkboxes** (membrane, nuclei raw, cell labels, nuclei labels) and a label **opacity** slider. Run §4 with `WRITE=True` first so it reads the saved volumes (it falls back to the in-memory arrays otherwise). Labels render instantly; tick the raw channels to overlay them — those read from the tiles, so zoom into a region first if a full slice feels slow.

In [ ]:
# --- Inline slice viewer (zero-install: matplotlib + ipywidgets; renders in THIS notebook output) ---
# Labels are memory-mapped (lazy); each raw slice is read on demand off the
# tiles (only the viewed crop). Render logic audit-verified (XY/XZ/YZ + zoom + overlay + edge cases).
try:
    get_ipython().run_line_magic("matplotlib", "inline")   # ensure inline backend (no-op outside IPython)
except Exception:
    pass
import matplotlib.pyplot as plt
from ipywidgets import (interactive_output, HBox, VBox, Dropdown, IntSlider, FloatSlider,
                        SelectionSlider, Checkbox)
from IPython.display import display
from skimage.segmentation import find_boundaries

MAX_DISPLAY_PX = 2000   # cap the rendered image's largest side (raise it to render higher-res)

# 1) labels: prefer the saved reconcile-dir files (memmap = lazy), else the in-memory arrays
recon_dir = CELL_RUN_DIR / "reconcile"
_cp, _npth = recon_dir / "cleanup_cell_labels.npy", recon_dir / "cleanup_nuclei_labels.npy"
if _cp.exists() and _npth.exists():
    CELLS = np.load(str(_cp), mmap_mode="r"); NUC = np.load(str(_npth), mmap_mode="r")
    print(f"labels: memory-mapped from {recon_dir}")
elif "out_cells" in globals() and "out_nuclei" in globals():
    CELLS, NUC = out_cells, out_nuclei
    print("labels: saved files not found (run §4 with WRITE=True) -> using the in-memory arrays")
else:
    raise SystemExit("no consensus found: run §2 (and §4 WRITE=True), or fix the reconcile/ path")
ZZ, YY, XX = CELLS.shape
print(f"volume dims (Z, Y, X) = {(ZZ, YY, XX)}")

# per-view display metadata: which volume axis is rows(vertical) / cols(horizontal) / the slice, and
# the physical pixel aspect (rows-spacing / cols-spacing) so ortho (XZ/YZ) aren't squished. Set
# VOXEL_SIZE_UM=(z,y,x) in §1 for physically-correct proportions; else voxels are treated as cubes.
_SP = tuple(float(v) for v in VOXEL_SIZE_UM) if VOXEL_SIZE_UM else (1.0, 1.0, 1.0)   # (z, y, x)
_AX = {"XY": ("Y", "X", "Z"), "XZ": ("Z", "X", "Y"), "YZ": ("Z", "Y", "X")}          # (rows, cols, slice)
_ASPECT = {"XY": _SP[1] / _SP[2], "XZ": _SP[0] / _SP[2], "YZ": _SP[0] / _SP[1]}       # rows/cols spacing

# 2) raw channels — lazy per-slice read off the tiles; best-effort (labels still work without it)
RAW = {}
try:
    from packages.epicycle_scaledseg.reconcile import load_geometry, resolve_channel_layout, read_global_roi
    _geo = load_geometry(CELL_TILES); _lay = resolve_channel_layout(CELL_TILES)
    _nch = int(_geo["channels"])
    _frame = (int(_geo["depth"]), int(_geo["full"][0]), int(_geo["full"][1]))
    if _frame != (ZZ, YY, XX):        # co-registration guard: raw ROIs are built from label dims
        raise ValueError(f"label shape {(ZZ, YY, XX)} != tile frame {_frame} (not co-registered)")
    _nuc = int(NUCLEI_CHANNEL) if NUCLEI_CHANNEL is not None else _lay.get("nucleus_channel")
    _sig = _lay.get("signal_channel")
    if _sig is None and _nuc is not None and _nch == 2:
        _sig = 1 - int(_nuc)
    _cand = {"membrane": _sig, "nuclei_raw": _nuc}
    RAW = {k: int(v) for k, v in _cand.items() if v is not None and 0 <= int(v) < _nch}
    _skip = {k: v for k, v in _cand.items() if k not in RAW}
    print(f"raw channels available: {RAW}" + (f"  (skipped invalid {_skip})" if _skip else "")
          + "  (membrane -> green, nuclei -> magenta)")
except Exception as e:
    print(f"raw channels unavailable ({type(e).__name__}: {e}) -> labels only")

# 3) stable random colours for the label overlays (large LUT -> few colour collisions)
_LUT_C = np.random.default_rng(0).random((50000, 3))
_LUT_N = np.random.default_rng(1).random((50000, 3))
_chan_cache = {}

def _norm(a, lo_p=1.0, hi_p=99.7):                  # gentle white point -> less over-saturation
    a = np.asarray(a, np.float32)
    lo, hi = np.percentile(a, lo_p), np.percentile(a, hi_p)
    return np.clip((a - lo) / (hi - lo + 1e-6), 0, 1) if hi > lo else np.zeros_like(a)

def _bounds(view, rowf, colf):
    (r0, r1), (c0, c1) = rowf, colf
    R = YY if view == "XY" else ZZ                 # rows axis length
    C = YY if view == "YZ" else XX                 # cols axis length
    a0 = min(int(r0 * R), R - 1); a1 = min(max(int(r1 * R), a0 + 1), R)   # clamp start in-bounds
    b0 = min(int(c0 * C), C - 1); b1 = min(max(int(c1 * C), b0 + 1), C)
    s = max(1, -(-max(a1 - a0, b1 - b0) // MAX_DISPLAY_PX))               # CEIL stride: cap actually holds
    return a0, a1, b0, b1, s

def _lab_slice(vol, view, idx, b):
    a0, a1, b0, b1, s = b
    if view == "XY": return np.asarray(vol[min(idx, ZZ - 1), a0:a1:s, b0:b1:s])
    if view == "XZ": return np.asarray(vol[a0:a1:s, min(idx, YY - 1), b0:b1:s])
    return np.asarray(vol[a0:a1:s, b0:b1:s, min(idx, XX - 1)])           # YZ -> (Z, Y)

def _raw_slice(ch, view, idx, b):
    a0, a1, b0, b1, s = b
    if view == "XY":   roi, ax = (min(idx, ZZ - 1), min(idx, ZZ - 1) + 1, a0, a1, b0, b1), 0
    elif view == "XZ": roi, ax = (a0, a1, min(idx, YY - 1), min(idx, YY - 1) + 1, b0, b1), 1
    else:              roi, ax = (a0, a1, b0, b1, min(idx, XX - 1), min(idx, XX - 1) + 1), 2
    crop = np.squeeze(np.asarray(read_global_roi(CELL_TILES, tuple(int(v) for v in roi), ch)), axis=ax)
    return crop[::s, ::s]

def _channels(view, idx, rowf, colf, membrane, nuclei_raw, b):
    # returns (green_src, magenta_src) normalized [0,1] 2D arrays or None; cached WITHOUT brightness
    key = (view, idx, tuple(rowf), tuple(colf), bool(membrane), bool(nuclei_raw))
    if key in _chan_cache:                          # reuse across brightness / label / alpha changes
        return _chan_cache[key]
    green = _norm(_raw_slice(RAW["membrane"], view, idx, b)) if (membrane and "membrane" in RAW) else None
    mag   = _norm(_raw_slice(RAW["nuclei_raw"], view, idx, b)) if (nuclei_raw and "nuclei_raw" in RAW) else None
    _chan_cache.clear(); _chan_cache[key] = (green, mag)
    return green, mag

def _render(view, idx, rowf, colf, membrane, nuclei_raw, cells, nuclei_lab, cell_outline, brightness, alpha):
    a0, a1, b0, b1, s = _bounds(view, rowf, colf); b = (a0, a1, b0, b1, s)
    hh, ww = len(range(a0, a1, s)), len(range(b0, b1, s))               # shape without touching CELLS
    img = np.zeros((hh, ww, 3), np.float32)
    green, mag = _channels(view, idx, rowf, colf, membrane, nuclei_raw, b)
    if green is not None:
        img[..., 1] = np.clip(green * brightness, 0, 1)                  # membrane -> GREEN
    if mag is not None:
        mm = np.clip(mag * brightness, 0, 1)
        img[..., 0] = np.maximum(img[..., 0], mm); img[..., 2] = np.maximum(img[..., 2], mm)  # -> MAGENTA
    if cells:                                                           # cells: outline OR solid
        cl = _lab_slice(CELLS, view, idx, b)
        if cell_outline:
            mask = find_boundaries(cl, mode="inner")
            if not mask.any() and (cl > 0).any():
                mask = cl > 0                     # crop is one whole cell -> draw solid (not invisible)
        else:
            mask = cl > 0
        img[mask] = (1 - alpha) * img[mask] + alpha * _LUT_C[cl[mask] % len(_LUT_C)]
    if nuclei_lab:                                                      # nuclei: solid
        nl = _lab_slice(NUC, view, idx, b); nm = nl > 0
        img[nm] = (1 - alpha) * img[nm] + alpha * _LUT_N[nl[nm] % len(_LUT_N)]
    return np.clip(img, 0, 1)

def _show(view, idx, zoom, cx, cy, membrane, nuclei_raw, brightness, cells, cell_outline, nuclei_lab, alpha):
    w = 1.0 / zoom                                                      # window fraction; pan to (cx,cy)
    r0 = min(max(cy - w / 2, 0.0), 1.0 - w); c0 = min(max(cx - w / 2, 0.0), 1.0 - w)
    perp = {"XY": ZZ, "XZ": YY, "YZ": XX}[view]; rax, cax, sax = _AX[view]
    img = _render(view, idx, (r0, r0 + w), (c0, c0 + w), membrane, nuclei_raw,
                  cells, nuclei_lab, cell_outline, brightness, alpha)
    fig, ax = plt.subplots(figsize=(9, 9)); ax.imshow(img, aspect=_ASPECT[view]); ax.axis("off")
    ax.set_title(f"{view}   {sax}-slice {min(idx, perp - 1)}/{perp - 1}   |   "
                 f"pan y = {rax} (vertical), pan x = {cax} (horizontal)   |   zoom {zoom}x")
    plt.show(); plt.close(fig)

# 4) widgets — grouped: navigate | zoom/pan | channels | labels. `slice` spans the current view's
#    perpendicular axis (resized on view change); zoom magnifies, cx/cy pan (only matter when zoomed).
_PERP = {"XY": ZZ, "XZ": YY, "YZ": XX}
w_view = Dropdown(options=["XY", "XZ", "YZ"], value="XY", description="view")
w_idx  = IntSlider(ZZ // 2, 0, ZZ - 1, 1, description="slice", continuous_update=False)
w_zoom = SelectionSlider(options=[1, 2, 4, 8, 16, 32], value=1, description="zoom")
w_cx   = FloatSlider(0.5, min=0, max=1, step=0.02, description="pan x", continuous_update=False)
w_cy   = FloatSlider(0.5, min=0, max=1, step=0.02, description="pan y", continuous_update=False)
w_mem  = Checkbox(value=False, description="membrane (green)")
w_nraw = Checkbox(value=False, description="nuclei (magenta)")
w_bri  = FloatSlider(1.0, min=0.2, max=4.0, step=0.1, description="brightness", continuous_update=False)
w_cell = Checkbox(value=True, description="cell labels")
w_out  = Checkbox(value=True, description="cell outlines")
w_nlab = Checkbox(value=True, description="nuclei labels")
w_alpha = FloatSlider(0.6, min=0, max=1, step=0.05, description="label opacity", continuous_update=False)

def _on_view(change):
    new_max = _PERP[change["new"]] - 1
    w_idx.max = max(w_idx.max, new_max)   # grow first so value is never transiently > max
    w_idx.value = new_max // 2            # jump to the middle plane of the new axis
    w_idx.max = new_max                   # shrink to the real per-view range
w_view.observe(_on_view, names="value")

_out_area = interactive_output(_show, dict(
    view=w_view, idx=w_idx, zoom=w_zoom, cx=w_cx, cy=w_cy, membrane=w_mem, nuclei_raw=w_nraw,
    brightness=w_bri, cells=w_cell, cell_outline=w_out, nuclei_lab=w_nlab, alpha=w_alpha))
display(VBox([HBox([w_view, w_idx]),
             HBox([w_zoom, w_cx, w_cy]),
             HBox([w_mem, w_nraw, w_bri]),
             HBox([w_cell, w_out, w_nlab, w_alpha])]), _out_area)
print("On XY: pan y = Y, pan x = X. On XZ/YZ the VERTICAL axis is Z (the title spells it out per view).")
print("Membrane=green, nuclei=magenta; brightness fixes over-saturation. Cells outline (untick for solid); nuclei solid.")
print("Ortho (XZ/YZ) proportions use VOXEL_SIZE_UM if set in §1 (else voxels treated as cubes).")

labels: memory-mapped from <run>/epicycle_output_cell/reconcile
volume dims (Z, Y, X) = (266, 5631, 4709)
raw channels available: {'membrane': 1, 'nuclei_raw': 0}  (membrane -> green, nuclei -> magenta)


Output()

On XY: pan y = Y, pan x = X. On XZ/YZ the VERTICAL axis is Z (the title spells it out per view).
Membrane=green, nuclei=magenta; brightness fixes over-saturation. Cells outline (untick for solid); nuclei solid.
Ortho (XZ/YZ) proportions use VOXEL_SIZE_UM if set in §1 (else voxels treated as cubes).


## 6. Features + triage table  *(per cell + per nucleus)*

Reuses the engine's basic morphology features on the cleaned **cells** *and* **nuclei**, then joins each cell's cleanup **`status`** (clean / merged_cell / split_nucleus / recovered_nucleus / no_nucleus_strung_victim / no_nucleus_genuine) + nucleus aggregates (`n_nuclei`, `nucleus_voxels`, `nucleus_cell_frac`). `split_nucleus` = a cell that was **rescued** by a carve (its only nucleus is a fresh carved slice).

**Excluding carved nuclei from SHAPE stats — use `carved_nucleus_ids`, NOT `status`.** Both halves of a split are carved (and, with `SMOOTH_SPLIT_NUCLEI`, smoothed at the cut), so their *shape* is a plausibility estimate, not measured. The **majority** half keeps `status='clean'` (it genuinely has one good nucleus) — so filtering on `status` would let it through. The authoritative signal is the per-cell **`carved_nucleus_ids`** list (it names *both* halves' ids); drop those nucleus ids before any nuclear-shape quantification.

**What `status` can and cannot detect.** This is a *structural-error worklist*, **not** a complete bad-cell detector. `status` is derived from the **nucleus count after cleanup**, so it reliably flags: visible 2-nuclei merges, no-nucleus cells, orphans. It **misses**, by construction:
- **correlated deep merges** — two cells fused *and* their two nuclei also strung into one label (common in deep tissue, where the same attenuation kills both the wall and the nuclear split). These read `n_nuclei==1 → "clean"`. "Both features" is then *one* failed vote counted twice, not two independent votes.
- **merges hidden by the majority vote** — a real 2-cell merge whose second nucleus was assigned by majority vote to a *neighbour*; the merged cell then reads `clean` and the neighbour gets `no_nucleus_strung_victim` (pointing at the wrong object).

Morphology (`area`, `sphericity`, `nucleus_cell_frac`) catches the *grotesque* outliers but a compact 2-cell merge can look entirely normal. `nucleus_cell_frac` is a soft co-signal, **not a gate** (a low value flags lost-nucleus merges *and* legitimately large cells). So: **`status="clean"` means "1 nucleus after cleanup", not "good cell".** The one signal that would catch the invisible class — interior membrane/cadherin intensity (a missed wall shows as bright membrane *inside* the cell) — is not in this table.

In [ ]:
from packages.epicycle_scaledseg.reconcile import extract_label_features, assemble_cell_table, assemble_nucleus_table

# Per-label basic morphology features on the cleaned volumes (in-memory arrays -> staged to a temp .npy for
# the memmap pool), reusing the engine's feature core. Then join the cleanup `status` per cell.
print("extracting CELL features…", flush=True)
cell_feats = extract_label_features(out_cells,  spacing=VOXEL_SIZE_UM, n_workers=N_WORKERS,
                                    progress=True, desc="cell feats")
print("extracting NUCLEUS features…", flush=True)
nuc_feats  = extract_label_features(out_nuclei, spacing=VOXEL_SIZE_UM, n_workers=N_WORKERS,
                                    progress=True, desc="nuc feats")

cell_table = assemble_cell_table(triage, cell_feats, nuc_feats)   # 1 row/cell + status + nucleus aggs
nuc_table  = assemble_nucleus_table(nuc_feats, triage)            # 1 row/nucleus + owning cell_id

print(f"\ncell_table: {len(cell_table):,} cells x {cell_table.shape[1]} cols ; "
      f"nuc_table: {len(nuc_table):,} nuclei")
print("\nstatus counts (your triage worklist):")
print(cell_table['status'].value_counts().to_string())
cols = [c for c in ['label', 'status', 'n_nuclei', 'area', 'nucleus_voxels',
                    'nucleus_cell_frac', 'sphericity', 'aspect_ratio'] if c in cell_table.columns]
cell_table[cols].head(10)

## 7. Write the feature tables  *(set WRITE_FEATURES=True)*

In [ ]:
WRITE_FEATURES = False   # <-- set True to save the feature + triage tables
if WRITE_FEATURES:
    out_dir = CELL_RUN_DIR / "reconcile"; out_dir.mkdir(exist_ok=True)
    for name, df in [("cell", cell_table), ("nucleus", nuc_table)]:
        df.to_csv(out_dir / f"cleanup_{name}_features.csv", index=False)
        try:
            df.to_parquet(out_dir / f"cleanup_{name}_features.parquet", index=False)
        except Exception as e:
            print(f"  (parquet skipped for {name}: {e}; CSV written)")
    print(f"WROTE cleanup_cell_features + cleanup_nucleus_features (.csv + .parquet) to {out_dir}")
    print("triage in EpiCycle: filter `status` in {merged_cell, no_nucleus_strung_victim, "
          "no_nucleus_genuine} for the residual cells worth curating.")
else:
    print("WRITE_FEATURES=False — tables held in memory (cell_table / nuc_table). Set True to save.")